# Notebook 27 — Sequence-level CRISPR off-target scoring

PSF's existing F6 off-target detector flags *pathway-level*
unintended activations. This notebook demonstrates the complementary
F9 layer: scoring *sequence-level* candidate off-target sites with
Evo 2 (production) or the deterministic similarity + seed-match
fallback (tests).

Research use only. Not for clinical decision-making.

In [ ]:
import numpy as np
from pathway_subtyping.qc.offtarget_sequence import (
    SimilarityBackend, SimulatedEvo2Backend, Evo2OffTargetScorer,
    compare_backends, auroc,
)

guide = 'AAAAAAAACCCCCCCCGGGG'
candidates = {
    'on_target':        'AAAAAAAACCCCCCCCGGGG',  # perfect match
    'seed_preserved':   'AAAAAAAATTTTTTTTGGGG',  # preserved seed, mutated tail
    'seed_broken':      'TTTTTTTTCCCCCCCCGGGG',  # broken seed, preserved tail
    'full_mismatch':    'GGGGGGGGGGGGGGGGGGGG',
}
for label, backend in [('baseline', SimilarityBackend()), ('evo2', SimulatedEvo2Backend(seed_length=8))]:
    print(f'\n=== {label} ===')
    for row in backend.score(guide, candidates):
        print(f"  {row.site_id:15s} sim={row.similarity_score:.2f} func={row.functional_score:.2f} combined={row.combined_score:.2f}")

## Benchmark Evo 2 vs the similarity baseline

Roadmap acceptance: AUROC improves by >= 0.03 on a held-out
CRISPR panel where the true off-target label is seed-conservation.

In [ ]:
# Synthetic panel: positives preserve the seed; negatives break it.
rng = np.random.default_rng(0)
def mutate(seq, positions, rng):
    out = list(seq)
    for p in positions:
        orig = out[p]
        out[p] = rng.choice([b for b in 'ACGT' if b != orig])
    return ''.join(out)

seed = guide[:8]; tail = guide[8:]
cands, labels = {}, {}
for i in range(25):
    positions = rng.choice(len(tail), size=rng.integers(0, len(tail)//2 + 1), replace=False)
    cands[f'POS_{i}'] = seed + mutate(tail, positions, rng)
    labels[f'POS_{i}'] = 1
for i in range(25):
    positions = rng.choice(len(seed), size=rng.integers(2, len(seed) + 1), replace=False)
    cands[f'NEG_{i}'] = mutate(seed, positions, rng) + tail
    labels[f'NEG_{i}'] = 0

res = compare_backends(
    baseline=SimilarityBackend(),
    contender=SimulatedEvo2Backend(seed_length=8),
    guide=guide, candidates=cands, true_labels=labels,
)
print(res)

## See also
- PSF v0.6 roadmap — Phase 3 F9: [docs/roadmap-v06-codeberg.md](../../docs/roadmap-v06-codeberg.md)